# Cross-Domain Review Sentiment Classification

This notebook implements Option 2 (Review Sentiment Analysis) for ARTI 404. It trains a binary sentiment classifier (Positive/Negative) on Amazon product and IMDb movie reviews, then evaluates it on an in-domain held-out test and on a cross-domain Yelp restaurant test.

Experimental question: which method performs better on restaurant reviews when trained on product and movie reviews? The main judgment is Macro F1 on the Yelp test, with Accuracy reported alongside; the in-domain test is a supporting comparison only. Two methods are compared: TF-IDF + Logistic Regression (Method 1) and frozen MiniLM sentence embeddings + Logistic Regression (Method 2). Settings for both methods are selected on Development data only; neither test set is used for tuning.

## Setup

Import the libraries used across the notebook.

In [5]:
import platform
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sentence_transformers
import sklearn
import torch

Fix the random seed so splits, shuffling and model fitting are reproducible.

In [6]:
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

Define relative project paths and create the output folders if they do not already exist. The notebook is expected to run from the project root, so every path here is relative rather than a path tied to one member's machine.

In [7]:
PROJECT_ROOT = Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
RESULTS = PROJECT_ROOT / "results"
MODELS = PROJECT_ROOT / "models"

for path in (DATA_RAW, DATA_PROCESSED, RESULTS, MODELS):
    path.mkdir(parents=True, exist_ok=True)

Record the exact package versions used in this run, so the environment can be reproduced later.

In [8]:
print("Python:", platform.python_version())
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("torch:", torch.__version__)

Python: 3.12.10
numpy: 2.5.4
pandas: 3.1.1
scikit-learn: 1.9.1
sentence-transformers: 6.2.0
torch: 2.15.0+cpu


## Data

Download the UCI Sentiment Labelled Sentences archive into `data/raw/` if it is not already there, and extract it. The download is skipped when the three required files are already present; `__MACOSX` entries added by macOS zip tools are ignored everywhere below.

In [9]:
import datetime
import json
import urllib.request
import zipfile

DATA_URL = "https://archive.ics.uci.edu/static/public/331/sentiment+labelled+sentences.zip"
RAW_ZIP_PATH = DATA_RAW / "sentiment+labelled+sentences.zip"
DOWNLOAD_INFO_PATH = DATA_RAW / "download_info.json"
REQUIRED_FILES = ["amazon_cells_labelled.txt", "imdb_labelled.txt", "yelp_labelled.txt"]


def find_file(filename):
    matches = [p for p in DATA_RAW.rglob(filename) if "__MACOSX" not in p.parts]
    return matches[0] if matches else None


existing_files = {name: find_file(name) for name in REQUIRED_FILES}

if all(existing_files.values()):
    print("Required data files already present in data/raw/, skipping download.")
    if DOWNLOAD_INFO_PATH.exists():
        recorded = json.loads(DOWNLOAD_INFO_PATH.read_text())
        print("Recorded download date:", recorded["download_date"])
    else:
        print("Recorded download date: unknown (no download_info.json found).")
else:
    download_date = datetime.date.today().isoformat()
    urllib.request.urlretrieve(DATA_URL, RAW_ZIP_PATH)
    with zipfile.ZipFile(RAW_ZIP_PATH) as archive:
        archive.extractall(DATA_RAW)
    DOWNLOAD_INFO_PATH.write_text(
        json.dumps({"source": DATA_URL, "download_date": download_date}, indent=2)
    )
    existing_files = {name: find_file(name) for name in REQUIRED_FILES}
    print(f"Downloaded and extracted on {download_date}.")

print("\nFiles found in data/raw/ (excluding __MACOSX):")
for path in sorted(DATA_RAW.rglob("*")):
    if path.is_file() and "__MACOSX" not in path.parts:
        print(" -", path.relative_to(PROJECT_ROOT))

missing = [name for name, path in existing_files.items() if path is None]
if missing:
    raise FileNotFoundError(f"Required file(s) not found after download: {missing}")

Downloaded and extracted on 2026-11-11.

Files found in data/raw/ (excluding __MACOSX):
 - data\raw\download_info.json
 - data\raw\sentiment labelled sentences\.DS_Store
 - data\raw\sentiment labelled sentences\amazon_cells_labelled.txt
 - data\raw\sentiment labelled sentences\imdb_labelled.txt
 - data\raw\sentiment labelled sentences\readme.txt
 - data\raw\sentiment labelled sentences\yelp_labelled.txt
 - data\raw\sentiment+labelled+sentences.zip


Read each source file line by line and split on the last tab, since the label is always the final field and the review text itself may contain other whitespace. Lines that fail to parse are reported, not dropped silently. Cleaning is limited to Unicode normalisation and whitespace collapsing so negation words are preserved and the stored text is not lowercased.

In [10]:
import unicodedata


def clean_text(text: str) -> str:
    # NFKC normalises compatibility characters (e.g. full-width punctuation);
    # whitespace is then collapsed without touching negation words.
    normalized = unicodedata.normalize("NFKC", text)
    return " ".join(normalized.split())


def load_data(path: Path, source: str) -> tuple[pd.DataFrame, list[str]]:
    rows = []
    invalid_lines = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, raw_line in enumerate(handle, start=1):
            line = raw_line.rstrip("\n").rstrip("\r")
            if not line.strip():
                invalid_lines.append(f"{source} line {line_number}: empty line")
                continue
            if "\t" not in line:
                invalid_lines.append(f"{source} line {line_number}: no tab separator")
                continue
            text_part, _, label_part = line.rpartition("\t")
            text = clean_text(text_part)
            label_part = label_part.strip()
            if label_part not in ("0", "1"):
                invalid_lines.append(
                    f"{source} line {line_number}: invalid label {label_part!r}"
                )
                continue
            if not text:
                invalid_lines.append(f"{source} line {line_number}: empty text")
                continue
            rows.append(
                {
                    "example_id": f"{source}_{line_number:04d}",
                    "source": source,
                    "text": text,
                    "label": int(label_part),
                }
            )
    return pd.DataFrame(rows), invalid_lines

Load all three sources and collect any invalid lines found during parsing.

In [11]:
FILENAME_BY_SOURCE = {
    "amazon": "amazon_cells_labelled.txt",
    "imdb": "imdb_labelled.txt",
    "yelp": "yelp_labelled.txt",
}

frames = []
all_invalid_lines = []
for source, filename in FILENAME_BY_SOURCE.items():
    frame, invalid_lines = load_data(existing_files[filename], source)
    frames.append(frame)
    all_invalid_lines.extend(invalid_lines)

raw_df = pd.concat(frames, ignore_index=True)

print(f"Loaded {len(raw_df)} valid rows from {len(FILENAME_BY_SOURCE)} source files.")
print(f"Invalid lines found: {len(all_invalid_lines)}")
for entry in all_invalid_lines:
    print(" -", entry)

Loaded 3000 valid rows from 3 source files.
Invalid lines found: 0


Counts by source and label, before any duplicate handling.

In [12]:
count_table = (
    raw_df.groupby(["source", "label"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: "negative_0", 1: "positive_1"})
)
count_table["total"] = count_table["negative_0"] + count_table["positive_1"]
count_table

label,negative_0,positive_1,total
source,,,
amazon,500,500,1000
imdb,500,500,1000
yelp,500,500,1000
